# Bird's eye view of all the measured points
In this notebook we'll try to categorize all the measured points we have.

In [6]:
import glob
import re
from th_ldv import dataIo
from th_ldv.utilities import (extract_date_from_filepath, filter_files_by_categories, filter_files, FILES_PATTERNS, 
get_corresponding_ffid_file)
import pandas as pd
import numpy as np


ImportError: cannot import name 'plot_helper' from partially initialized module 'th_ldv' (most likely due to a circular import) (C:\Users\Lior\Documents\GitHub\TH_ldv\src\th_ldv\__init__.py)

first we'll map all measured location, for each profile there is a filder inside adate indicating folder, thus we have something of this sort:

`/lda/YYYY_MM_DD/TH_*`

I'll map all those folder, out of them there are a few special ones, folder for wider measurments, for span checking etc.
Thus the categories will be,
- main_measurments: close and far field measurments at differents heights and span location to map different cells. mainly at y=-50,-25,0,25,50 and z=65,95,110,125,180,220
- long duration measurements: few locations in which we measured for long time duration, 30 minn indicated by `*_long_*`
- wider measurments: lower height we performed measurments at wider `Y` span specifically at y=-100,-75,75,100. measurment time was 20 min for each point.
- we also preformed velocity magnitude span check. LDA system may cut opp measurments that are out of the defined span. thus we did some test to determine wether we lost extreme values that are meaningfull
- Z profiles, measured for 3 min at each location devided to lower height with higher resolution and height heights with lower spatial resolution.

Below we'll see the full list of measured files, in each folder there are a set of measured points.

In [2]:
measured_files = glob.glob('../data/raw/LDA/*/*TH*', recursive=True)
measured_files = [file.replace('\\','/') for file in measured_files]

We'll start filtering out all the testing cases in order to create the main folder list,

In [3]:

def folder_list_to_tbl(folder_lst):    
    files_dic={'file_path':[],
               'sample_time':[],
               'dt_max':[],
               'length':[],
               'x':[],
               'y':[],
               'z':[],
              }
    for folder in folder_lst:
        files_lst=glob.glob(folder+'/**/V*.txt', recursive=True)
        for file in files_lst:
            data=dataIo.load_lda_point(file)
            files_dic['file_path'].append(file.removeprefix('../data/'))
            files_dic['sample_time'].append(float(data.time.max().data*1e-3))
            files_dic['length'].append(len(data.time))
            files_dic['dt_max'].append(float(np.max(data.time.diff(dim='time'))))
            files_dic['x'].append(float(data.x.data))
            files_dic['y'].append(float(data.y.data))
            files_dic['z'].append(float(data.z.data))
    file_df=pd.DataFrame(files_dic)
    return file_df

# main measurments
In this files lower heights were measured for 20 min higher heights measured for 10 min and both far fiels and close field cells were maped.
We should get 120 measured points, taking the max samling time at each points in case we have duplicates.

In [4]:
main_data_folders= filter_files_by_categories(measured_files, FILES_PATTERNS['main'])
main_df=folder_list_to_tbl(main_data_folders)

In [5]:
main_df

,file_path,sample_time,dt_max,length,x,y,z
0,raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...,599.993632,103.25310,152287,997.0,-25.0,65.0
1,raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...,599.998596,141.55830,179495,997.0,-25.0,95.0
2,raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...,599.995795,88.13620,240025,997.0,-25.0,110.0
3,raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...,600.002765,81.16495,262546,997.0,-25.0,125.0
4,raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...,599.986710,42.69515,318611,997.0,-25.0,180.0
...,...,...,...,...,...,...,...
137,raw/LDA/2024_11_07/TH_C5Z220_Y_lpm_075_40hz\VE...,599.993143,76.60365,147477,482.0,-50.0,220.0
138,raw/LDA/2024_11_07/TH_C5Z220_Y_lpm_075_40hz\VE...,599.997506,78.48445,147011,482.0,-25.0,220.0
139,raw/LDA/2024_11_07/TH_C5Z220_Y_lpm_075_40hz\VE...,599.991987,78.76210,133694,482.0,0.0,220.0
140,raw/LDA/2024_11_07/TH_C5Z220_Y_lpm_075_40hz\VE...,599.984007,76.25100,131396,482.0,25.0,220.0


In [6]:
main_filtered = filter_files(main_df)
main_filtered

,file_path,sample_time,dt_max,length,x,y,z
92,raw/LDA/2024_11_05/TH_C4Z65_Y_lpm_075_40hz\VEL...,899.968456,103.04365,193532,407.0,-50.0,65.0
97,raw/LDA/2024_11_05/TH_C4Z95_Y_lpm_075_40hz\VEL...,899.988071,105.94970,225692,407.0,-50.0,95.0
81,raw/LDA/2024_11_05/TH_C4Z110_Y_lpm_075_40hz\VE...,900.013013,91.03660,262826,407.0,-50.0,110.0
86,raw/LDA/2024_11_05/TH_C4Z125_Y_lpm_075_40hz\VE...,899.978999,89.97125,277106,407.0,-50.0,125.0
102,raw/LDA/2024_11_06/TH_C4Z180_Y_lpm_075_40hz\VE...,599.992980,78.83385,158948,407.0,-50.0,180.0
...,...,...,...,...,...,...,...
59,raw/LDA/2024_11_03/TH_F1Z95_Y_lpm_400_40hz\VEL...,899.970076,82.83905,247776,1072.0,50.0,95.0
34,raw/LDA/2024_11_03/TH_F1Z110_Y_lpm_400_40hz\VE...,899.999519,82.50215,300028,1072.0,50.0,110.0
39,raw/LDA/2024_11_03/TH_F1Z125_Y_lpm_400_40hz\VE...,899.991309,86.14685,288106,1072.0,50.0,125.0
44,raw/LDA/2024_11_03/TH_F1Z180_Y_lpm_400_40hz\VE...,599.997067,58.98705,225725,1072.0,50.0,180.0


In [7]:
def get_filtered_off_rows(df, filtered_df):
    # Convert 'data_path' columns to sets for both DataFrames
    main_set = set(df['file_path'])
    filtered_set = set(filtered_df['file_path'])
    # Find the difference (files in main_df but not in filtered_df)
    difference = main_set - filtered_set
    # Filter the rows in main_df that are not in filtered_df
    filtered_off_rows = df[df['file_path'].isin(difference)]
    return filtered_off_rows


In [8]:
filtered_off_main = get_filtered_off_rows(main_df, main_filtered)
print("Filtered off rows from main_df:")
print(filtered_off_main)

Filtered off rows from main_df:
                                            file_path  sample_time  \
0   raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...   599.993632   
1   raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...   599.998596   
2   raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...   599.995795   
3   raw/LDA/2024_10_31/TH_F0Y-25_Z_lpm_400_40hz\VE...   600.002765   
6   raw/LDA/2024_10_31/TH_F0Y-50_Z_lpm_400_40hz\VE...   599.987338   
7   raw/LDA/2024_10_31/TH_F0Y-50_Z_lpm_400_40hz\VE...   599.994491   
8   raw/LDA/2024_10_31/TH_F0Y-50_Z_lpm_400_40hz\VE...   599.991708   
9   raw/LDA/2024_10_31/TH_F0Y-50_Z_lpm_400_40hz\VE...   599.975083   
12  raw/LDA/2024_10_31/TH_F0Y0_Z_lpm_400_40hz\VEL_...   599.996400   
13  raw/LDA/2024_10_31/TH_F0Y0_Z_lpm_400_40hz\VEL_...   600.011104   
14  raw/LDA/2024_10_31/TH_F0Y0_Z_lpm_400_40hz\VEL_...   599.983053   
15  raw/LDA/2024_10_31/TH_F0Y0_Z_lpm_400_40hz\VEL_...   599.994073   
18  raw/LDA/2024_10_31/TH_F0Y25_Z_lpm_400_40hz\VEL...   59

In [9]:
#check duplicates
grouped = main_filtered.groupby(['x','y','z']).size()
main_filtered.set_index(['x','y','z']).loc[grouped[grouped > 1].index].reset_index()


,x,y,z,file_path,sample_time,dt_max,length


In [10]:
main_filtered.to_csv('../data/main_files_lst.csv')
filtered_off_main.to_csv('../data/filtered_off_main.csv')

In [11]:
main_filtered

,file_path,sample_time,dt_max,length,x,y,z
92,raw/LDA/2024_11_05/TH_C4Z65_Y_lpm_075_40hz\VEL...,899.968456,103.04365,193532,407.0,-50.0,65.0
97,raw/LDA/2024_11_05/TH_C4Z95_Y_lpm_075_40hz\VEL...,899.988071,105.94970,225692,407.0,-50.0,95.0
81,raw/LDA/2024_11_05/TH_C4Z110_Y_lpm_075_40hz\VE...,900.013013,91.03660,262826,407.0,-50.0,110.0
86,raw/LDA/2024_11_05/TH_C4Z125_Y_lpm_075_40hz\VE...,899.978999,89.97125,277106,407.0,-50.0,125.0
102,raw/LDA/2024_11_06/TH_C4Z180_Y_lpm_075_40hz\VE...,599.992980,78.83385,158948,407.0,-50.0,180.0
...,...,...,...,...,...,...,...
59,raw/LDA/2024_11_03/TH_F1Z95_Y_lpm_400_40hz\VEL...,899.970076,82.83905,247776,1072.0,50.0,95.0
34,raw/LDA/2024_11_03/TH_F1Z110_Y_lpm_400_40hz\VE...,899.999519,82.50215,300028,1072.0,50.0,110.0
39,raw/LDA/2024_11_03/TH_F1Z125_Y_lpm_400_40hz\VE...,899.991309,86.14685,288106,1072.0,50.0,125.0
44,raw/LDA/2024_11_03/TH_F1Z180_Y_lpm_400_40hz\VE...,599.997067,58.98705,225725,1072.0,50.0,180.0


# wide locations
in order to check wider points $y=\pm 75$ and $y= \pm 100$. at elevation up to 180

In [12]:
wide_data_folders= filter_files_by_categories(measured_files, FILES_PATTERNS['wide'])
wide_df=folder_list_to_tbl(wide_data_folders)

In [13]:
wide_filtered=filter_files(wide_df)


In [14]:
# Count occurrences of each (x, y, z) combination
grouped = wide_df.groupby(['x', 'y', 'z']).size()

# Filter for combinations that appear more than once
duplicates = grouped[grouped > 1].index
# Filter the original DataFrame for rows matching these combinations
result = wide_df[wide_df.set_index(['x', 'y', 'z']).index.isin(duplicates)]
print(result)

                                            file_path  sample_time    dt_max  \
16  raw/LDA/2024_11_11/TH_C5Z180_Y_wide_lpm_075_40...   899.999234  53.96295   
17  raw/LDA/2024_11_11/TH_C5Z180_Y_wide_lpm_075_40...   899.995773  63.41625   
18  raw/LDA/2024_11_11/TH_C5Z180_Y_wide_lpm_075_40...   899.991505  67.75900   
19  raw/LDA/2024_11_11/TH_C5Z180_Y_wide_lpm_075_40...   899.993403  76.56815   
44  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.996544  64.09125   
45  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.989736  69.79990   
46  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.996629  89.88200   
47  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.993497  75.54325   

    length      x      y      z  
16  446413  482.0 -100.0  180.0  
17  427675  482.0  -75.0  180.0  
18  372002  482.0   75.0  180.0  
19  359455  482.0  100.0  180.0  
44  217966  482.0 -100.0  180.0  
45  209750  482.0  -75.0  180.0  
46  194302  482.0   75.0  180.0  
47  189

In [15]:
filtered_off_wide = get_filtered_off_rows(wide_df,wide_filtered)
print(filtered_off_wide)

                                            file_path  sample_time    dt_max  \
44  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.996544  64.09125   
45  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.989736  69.79990   
46  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.996629  89.88200   
47  raw/LDA/2024_11_12/TH_C5Z180_Y_wide_lpm_075_40...   599.993497  75.54325   

    length      x      y      z  
44  217966  482.0 -100.0  180.0  
45  209750  482.0  -75.0  180.0  
46  194302  482.0   75.0  180.0  
47  189978  482.0  100.0  180.0  


In [16]:
wide_filtered.to_csv('../data/wide_files_lst.csv')

# long measurments
locations in which we measured for long sampling time ( 30 min)

In [17]:
long_data_folders= filter_files_by_categories(measured_files, FILES_PATTERNS['long'])
long_df=folder_list_to_tbl(long_data_folders)

In [18]:
long_filtered= filter_files(long_df)
long_filtered

,file_path,sample_time,dt_max,length,x,y,z
5,raw/LDA/2024_11_13/TH_C4Z180Y-50_long_lpm_075_...,1799.997402,61.26450,884044,407.0,-50.0,180.0
2,raw/LDA/2024_11_05/TH_C4Z65Y0_long_lpm_075_40h...,1799.976773,148.84310,238907,407.0,0.0,65.0
3,raw/LDA/2024_11_05/TH_C4Z95Y0_long_lpm_075_40h...,1799.986534,142.14295,314833,407.0,0.0,95.0
1,raw/LDA/2024_11_05/TH_C4Z125Y0_long_lpm_075_40...,1799.986908,99.31895,643437,407.0,0.0,125.0
6,raw/LDA/2024_11_13/TH_C4Z95Y25_long_lpm_075_40...,1799.990955,117.88110,430808,407.0,25.0,95.0
7,raw/LDA/2024_11_13/TH_C5Z95Y50_long_lpm_075_40...,1799.996600,99.55555,443769,482.0,50.0,95.0
0,raw/LDA/2024_11_04/TH_F0Z65Y0_long_lpm_400_40h...,1799.971237,175.82570,222631,997.0,0.0,65.0
4,raw/LDA/2024_11_05/TH_F0Z95Y0_long_lpm_400_40h...,1799.999055,164.57720,306814,997.0,0.0,95.0
8,raw/LDA/2024_11_13/TH_F0Z95Y25_long_lpm_400_40...,1799.994499,141.67135,436059,997.0,25.0,95.0


In [19]:
filtered_off_long = get_filtered_off_rows(long_df, long_filtered)
print(filtered_off_long)

Empty DataFrame
Columns: [file_path, sample_time, dt_max, length, x, y, z]
Index: []


In [20]:
long_df.to_csv('../data/long_files_lst.csv')

# Span check
for velocity cut off corrections


In [27]:
span_check_path = glob.glob('../data/raw/LDA/*/*TH*Span*', recursive=True)
span_check_path = glob.glob(span_check_path[0][:-18]+'*', recursive=True)
span_check_path

['../data/raw/LDA\\2024_11_17\\TH_F0Y50_Z_lpm_000_40hz_SpanCheck']

In [28]:
span_check_df=folder_list_to_tbl(span_check_path)

In [23]:
grouped = span_check_df.groupby(['x', 'y', 'z']).size()
span_check=span_check_df.set_index(['x','y','z']).loc[grouped[grouped > 1].index].reset_index()
span_check

,x,y,z,file_path,sample_time,dt_max,length


In [29]:
span_check_df.to_csv('../data/span_check_files.csv')

# z profile ldv ffid
I need to filter the different profiles, some are bad and were repeated, some were taken without ethan etc

In [30]:
z_profile_data_folders= filter_files_by_categories(measured_files, FILES_PATTERNS['z_profiles'])
z_profile_df=folder_list_to_tbl(z_profile_data_folders)

In [31]:
z_profile_df[z_profile_df.dt_max>300]

,file_path,sample_time,dt_max,length,x,y,z


In [32]:
z_profile_filtered = filter_files(z_profile_df, choose_max_time= False )

In [33]:
z_profile_filtered.groupby(['x','y']).apply(list)


C:\Users\Lior\AppData\Local\Temp\ipykernel_1520\315536863.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  z_profile_filtered.groupby(['x','y']).apply(list)


x       y   
407.0   0.0     [file_path, sample_time, dt_max, length, x, y, z]
        25.0    [file_path, sample_time, dt_max, length, x, y, z]
        50.0    [file_path, sample_time, dt_max, length, x, y, z]
482.0   0.0     [file_path, sample_time, dt_max, length, x, y, z]
        25.0    [file_path, sample_time, dt_max, length, x, y, z]
        50.0    [file_path, sample_time, dt_max, length, x, y, z]
997.0   0.0     [file_path, sample_time, dt_max, length, x, y, z]
        25.0    [file_path, sample_time, dt_max, length, x, y, z]
        50.0    [file_path, sample_time, dt_max, length, x, y, z]
1072.0  0.0     [file_path, sample_time, dt_max, length, x, y, z]
        25.0    [file_path, sample_time, dt_max, length, x, y, z]
        50.0    [file_path, sample_time, dt_max, length, x, y, z]
dtype: object

In [34]:
#F1Y25 PROBLEMATIC MEASUREMENTS
indexes_to_drop = (z_profile_filtered['x'] == 1072.0) & (z_profile_filtered['y'] == 25.0) & (z_profile_filtered['z'] > 300.0)
z_profile_filtered = z_profile_filtered[~indexes_to_drop]  # Drop the rows that match the condition

In [35]:
#c4y50 PROBLEMATIC MEASUREMENTS
indexes_to_drop = (z_profile_filtered['x'] == 407.0) & (z_profile_filtered['y'] == 50.0) & (z_profile_filtered['z'] > 440.0)
z_profile_filtered = z_profile_filtered[~indexes_to_drop]  # Drop the rows that match the condition

In [36]:
#c5y25 PROBLEMATIC MEASUREMENTS
indexes_to_drop = (z_profile_filtered['x'] == 482.0) & (z_profile_filtered['y'] == 25.0) & (z_profile_filtered['z'] > 210.0)
z_profile_filtered = z_profile_filtered[~indexes_to_drop]  # Drop the rows that match the condition

In [37]:
z_profile_filtered.set_index(['x','y']).loc[482,25]['z']

C:\Users\Lior\AppData\Local\Temp\ipykernel_1520\3062003694.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  z_profile_filtered.set_index(['x','y']).loc[482,25]['z']


x      y   
482.0  25.0    150.0
       25.0    160.0
       25.0    170.0
       25.0    180.0
       25.0    190.0
       25.0    200.0
       25.0    210.0
       25.0     65.0
       25.0     70.0
       25.0     75.0
       25.0     80.0
       25.0     85.0
       25.0     90.0
       25.0     95.0
       25.0    100.0
       25.0    105.0
       25.0    110.0
       25.0    115.0
       25.0    120.0
       25.0    125.0
       25.0    130.0
       25.0    140.0
       25.0    150.0
       25.0    160.0
       25.0    170.0
       25.0    180.0
Name: z, dtype: float64

In [38]:
from TH_ffid_ldv.utilities import get_ldv_data_daily_order
for i, path in enumerate(z_profile_filtered.file_path):
    order = get_ldv_data_daily_order(path)
    if order[0] == order[1]:
        print(f"{path} \n has the same order {order[0]}/{order[1]}")
        z_profile_filtered = z_profile_filtered[~(z_profile_filtered.file_path == path)]  # Drop the rows that match the condition


raw/LDA/2024_11_07/TH_F0Y0_Zhigh_lpm_400_40hz\VEL_TH_F0Y0_Zhigh_lpm_400_40hz.000172.txt 
 has the same order 15/15
raw/LDA/2024_11_07/TH_F0Y0_Zlow_lpm_400_40hz\VEL_TH_F0Y0_Zlow_lpm_400_40hz.000198.txt 
 has the same order 26/26
raw/LDA/2024_11_10/TH_F1Y0_Zhigh_lpm_400_40hz\VEL_TH_F1Y0_Zhigh_lpm_400_40hz.000240.txt 
 has the same order 15/15
raw/LDA/2024_11_10/TH_F1Y0_Zlow_lpm_400_40hz\VEL_TH_F1Y0_Zlow_lpm_400_40hz.000225.txt 
 has the same order 26/26
raw/LDA/2024_11_13/TH_C4Y0_Zlnh_lpm_075_40hz\VEL_TH_C4Y0_Zlnh_lpm_075_40hz.000351.txt 
 has the same order 37/37
raw/LDA/2024_11_14/TH_C5Y0_Zhigh_lpm_075_40hz\VEL_TH_C5Y0_Zhigh_lpm_075_40hz.000391.txt 
 has the same order 32/32
raw/LDA/2024_11_17/TH_C5Y0_Zlow_lpm_075_40hz\VEL_TH_C5Y0_Zlow_lpm_075_40hz.000359.txt 
 has the same order 8/8
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000430.txt 
 has the same order 37/37
raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz\VEL_TH_F0Y50_Zhigh_lpm_400_40hz.000456.txt 

In [39]:
for i, path in enumerate(z_profile_filtered.file_path):
    if '2024_11_17/TH_F0Y50_Zlnh_lpm_400' in path:
        if z_profile_filtered[(z_profile_filtered.file_path == path)]['z'].iloc[0]>200:
            z_profile_filtered = z_profile_filtered[~(z_profile_filtered.file_path == path)]  # Drop the rows that match the condition
            print(f"{path} \n has been removed from list")

raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000415.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000416.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000417.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000418.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000419.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000420.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000421.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_lpm_400_40hz.000422.txt 
 has been removed from list
raw/LDA/2024_11_17/TH_F0Y50_Zlnh_lpm_400_40hz\VEL_TH_F0Y50_Zlnh_

In [40]:
z_profile_filtered[(z_profile_filtered.x==997)&(z_profile_filtered.y==50)&(z_profile_filtered.z>200)]['file_path']

199    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
200    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
201    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
202    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
203    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
204    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
205    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
206    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
207    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
208    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
209    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
210    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
211    raw/LDA/2024_11_18/TH_F0Y50_Zhigh_lpm_400_40hz...
Name: file_path, dtype: object

In [41]:
z_profile_filtered.to_csv('../data/z_profiles_ldv_ffid_lst.csv')

# repeated locations
files that were meared at the same locations in order to check for repeatability of the measurments.

In [42]:
concated=pd.concat([main_filtered,wide_filtered,long_filtered,z_profile_filtered])
grouped=concated.groupby(['x','y','z']).size()



In [43]:
repeated_locations=concated.set_index(['x','y','z']).loc[grouped[grouped > 1].index].reset_index()

In [44]:
repeated_locations.groupby(['x','y']).apply(list)

C:\Users\Lior\AppData\Local\Temp\ipykernel_1520\2813636639.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  repeated_locations.groupby(['x','y']).apply(list)


x       y    
407.0   -50.0    [x, y, z, file_path, sample_time, dt_max, length]
         0.0     [x, y, z, file_path, sample_time, dt_max, length]
         25.0    [x, y, z, file_path, sample_time, dt_max, length]
         50.0    [x, y, z, file_path, sample_time, dt_max, length]
482.0    0.0     [x, y, z, file_path, sample_time, dt_max, length]
         25.0    [x, y, z, file_path, sample_time, dt_max, length]
         50.0    [x, y, z, file_path, sample_time, dt_max, length]
997.0    0.0     [x, y, z, file_path, sample_time, dt_max, length]
         25.0    [x, y, z, file_path, sample_time, dt_max, length]
         50.0    [x, y, z, file_path, sample_time, dt_max, length]
1072.0   0.0     [x, y, z, file_path, sample_time, dt_max, length]
         25.0    [x, y, z, file_path, sample_time, dt_max, length]
         50.0    [x, y, z, file_path, sample_time, dt_max, length]
dtype: object

In [45]:
repeated_locations.to_csv('../data/repeated_locations_files.csv')

# Background measurments list

In [46]:
background_files_lst = filter_files_by_categories(measured_files, FILES_PATTERNS['background'], skip_lpm0= False)
background_df = folder_list_to_tbl(background_files_lst[:-2])

In [47]:
background_df

,file_path,sample_time,dt_max,length,x,y,z
0,raw/LDA/2024_10_31/TH_F0Y25_Z_lpm_000_40hz\VEL...,19.996981,23.20685,9260,997.0,25.0,220.0
1,raw/LDA/2024_11_04/TH_F0Z110Y50_lpm_000_40hz\V...,119.994540,84.24620,35475,997.0,50.0,110.0
2,raw/LDA/2024_11_04/TH_F0Z125Y50_lpm_000_40hz\V...,119.993562,52.53005,33931,997.0,50.0,125.0
3,raw/LDA/2024_11_04/TH_F0Z65_Y_lpm_000_40hz\VEL...,119.998274,56.43840,67158,997.0,-50.0,578.0
4,raw/LDA/2024_11_04/TH_F0Z95Y50_lpm_000_40hz\VE...,119.983881,99.21340,26402,997.0,50.0,95.0
5,raw/LDA/2024_11_05/TH_C4Z125Y0_lpm_000_40hz\VE...,119.997467,60.47900,41148,407.0,0.0,125.0
6,raw/LDA/2024_11_05/TH_C4Z95Y50_lpm_000_40hz\VE...,119.997036,76.12550,26946,407.0,50.0,95.0
7,raw/LDA/2024_11_06/TH_C4Z125Y25_lpm_000_40hz\V...,119.996600,107.84465,25532,407.0,25.0,125.0
8,raw/LDA/2024_11_07/TH_C5Z125Y-50_lpm_000_40hz\...,119.999046,72.29830,27992,482.0,-50.0,125.0
9,raw/LDA/2024_11_07/TH_C5Z180Y50_lpm_000_40hz\V...,119.991525,59.10935,28135,482.0,50.0,180.0


In [48]:
background_df.to_csv('../data/lpm_0_files.csv')

# Source Inlet velocuty profiles

In [49]:
inlet_files_lst = filter_files_by_categories(measured_files, FILES_PATTERNS['inlet'], skip_lpm0= False)
inlet_df = folder_list_to_tbl(inlet_files_lst)

In [50]:
inlet_df.to_csv('../data/inlet_files.csv')

# Additional F cell locations for velocity charactarization

In [51]:
velocity_files_lst = filter_files_by_categories(measured_files, FILES_PATTERNS['velocity'], skip_lpm0= False)
velocity_df = folder_list_to_tbl(velocity_files_lst)

In [52]:
velocity_df.to_csv('../data/velocity_files.csv')

# Y concentration profiles

In [54]:
y_files_lst = glob.glob("../data/raw/PC_LabView/20241030/2024_10_30_TH*-*.txt")
files_dic={'file_path':[],
			'sample_time':[],
			'dt_max':[],
			'length':[],
			'x':[],
			'y':[],
			'z':[],
			}
for file in y_files_lst:
	data=dataIo.load_ffid_data(file)
	files_dic['file_path'].append(file)
	files_dic['sample_time'].append(float(data.time.max().data*1e-3))
	files_dic['length'].append(len(data.time))
	files_dic['dt_max'].append(float(np.max(data.time.diff(dim='time'))))
	files_dic['x'].append(float(data.x.data))
	files_dic['y'].append(float(data.y.data))
	files_dic['z'].append(float(data.z.data))
file_df=pd.DataFrame(files_dic)
file_df

,file_path,sample_time,dt_max,length,x,y,z
0,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,179.998,2.0,90000,0.0,0.0,0.0
1,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,179.998,2.0,90000,0.0,0.0,0.0
2,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,179.998,2.0,90000,0.0,0.0,0.0
3,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,179.998,2.0,90000,0.0,0.0,0.0
4,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,179.998,2.0,90000,0.0,0.0,0.0
...,...,...,...,...,...,...,...
137,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,4.998,2.0,2500,0.0,0.0,0.0
138,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,4.998,2.0,2500,0.0,0.0,0.0
139,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,4.998,2.0,2500,0.0,0.0,0.0
140,../data/raw/PC_LabView/20241030\2024_10_30_TH_...,4.998,2.0,2500,0.0,0.0,0.0


In [55]:
file_df.to_csv('../data/y_spread_files.csv')